# BPE Tokenizer

This notebook implements a **byte-level Byte-Pair Encoding (BPE)** tokenizer
for the GPT project. It covers two classes:

- **Vocabulary** — bidirectional mapping between string tokens and integer IDs.
- **BPETokenizer** — trains merge rules on a text corpus, then encodes/decodes
  arbitrary strings (including Arabic UTF-8) to/from token ID sequences.

### How byte-level BPE works

1. Every input string is first converted to its raw UTF-8 bytes.
2. Each byte becomes an initial token using `<0xHH>` notation (256 base tokens).
3. During training, the most frequent adjacent byte-pair is merged into a new
   token. This repeats until the vocabulary reaches the target size.
4. Because we operate on bytes, multi-byte characters (Arabic, emoji, etc.)
   are never split within a single UTF-8 code point — the merge process
   naturally groups them.

Special tokens occupy IDs 0–3:

| Token | ID | Purpose |
|---|---|---|
| `<pad>` | 0 | Padding |
| `<bos>` | 1 | Beginning of sequence |
| `<eos>` | 2 | End of sequence |
| `<unk>` | 3 | Unknown / out-of-vocabulary |

In [ ]:
from __future__ import annotations

import json
import tempfile
import os
from collections import Counter
from typing import Any

## Vocabulary

A bidirectional mapping between string tokens and integer IDs.
Unknown tokens resolve to the `<unk>` special token (ID 3 by default).

| Method | Description |
|---|---|
| `add_token(token)` | Add a token and return its ID (no-op if already present) |
| `get_id(token)` | Return the ID for a token, or `<unk>` ID if not found |
| `get_token(token_id)` | Return the token for an ID, or `<unk>` if not found |
| `__len__()` | Return the vocabulary size |

In [ ]:
class Vocabulary:
    """Bidirectional mapping between string tokens and integer IDs.

    Unknown tokens resolve to the ``<unk>`` special token (ID 3 by default).
    """

    UNK_TOKEN = "<unk>"

    def __init__(self) -> None:
        self.token_to_id: dict[str, int] = {}
        self.id_to_token: dict[int, str] = {}

    def add_token(self, token: str) -> int:
        """Add *token* and return its integer ID.

        If the token is already present the existing ID is returned (no-op).
        """
        if token in self.token_to_id:
            return self.token_to_id[token]
        new_id = len(self.token_to_id)
        self.token_to_id[token] = new_id
        self.id_to_token[new_id] = token
        return new_id

    def get_id(self, token: str) -> int:
        """Return the ID for *token*, or the ``<unk>`` ID if not found."""
        if token in self.token_to_id:
            return self.token_to_id[token]
        return self.token_to_id.get(self.UNK_TOKEN, -1)

    def get_token(self, token_id: int) -> str:
        """Return the token for *token_id*, or ``<unk>`` if not found."""
        if token_id in self.id_to_token:
            return self.id_to_token[token_id]
        return self.UNK_TOKEN

    def __len__(self) -> int:
        """Return the number of tokens in the vocabulary."""
        return len(self.token_to_id)

## BPETokenizer

The main tokenizer class. It operates on UTF-8 bytes so that Arabic
(and any other multi-byte UTF-8) text is handled without splitting within
a single code point.

| Method | Description |
|---|---|
| `train(text)` | Learn BPE merge rules from a corpus |
| `encode(text)` | Encode a string to a list of token IDs |
| `decode(token_ids)` | Decode token IDs back to a string |
| `save(path)` | Save vocabulary and merges to JSON |
| `load(path)` | Load vocabulary and merges from JSON |

In [ ]:
class BPETokenizer:
    """Byte-Pair Encoding tokenizer operating on UTF-8 bytes.

    Special tokens ``<pad>``, ``<bos>``, ``<eos>``, ``<unk>`` occupy IDs 0-3.
    """

    SPECIAL_TOKENS: dict[str, int] = {
        "<pad>": 0,
        "<bos>": 1,
        "<eos>": 2,
        "<unk>": 3,
    }

    def __init__(self, vocab_size: int = 8000) -> None:
        """Initialise with a target vocabulary size.

        Args:
            vocab_size: Desired vocabulary size (including special tokens and
                the 256 base byte tokens).
        """
        self.vocab_size = vocab_size
        self.vocab = Vocabulary()
        self.merges: list[tuple[str, str]] = []

        # Seed special tokens first (IDs 0-3)
        for token, _expected_id in self.SPECIAL_TOKENS.items():
            self.vocab.add_token(token)

### Private Helpers

Low-level utilities used by the training and encoding methods.

| Helper | Purpose |
|---|---|
| `_byte_token(byte_val)` | Convert a byte value to `<0xHH>` notation |
| `_token_to_bytes(token)` | Convert a (possibly merged) token back to raw bytes |
| `_corpus_to_words(text)` | Split corpus into whitespace-delimited words as byte-token lists |
| `_count_pairs(words)` | Count adjacent token pairs across all words |
| `_apply_merge(words, pair, merged)` | Replace all occurrences of a pair in word lists |
| `_apply_merge_to_sequence(tokens, left, right)` | Merge adjacent tokens in a single sequence |

In [ ]:
@staticmethod
def _byte_token(byte_val: int) -> str:
    """Return the canonical string representation of a single byte.

    Uses ``<0xHH>`` notation so byte tokens never collide with real text
    or special tokens.
    """
    return f"<0x{byte_val:02X}>"


@staticmethod
def _token_to_bytes(token: str) -> bytes:
    """Convert a (possibly merged) token back to raw bytes."""
    result = bytearray()
    i = 0
    while i < len(token):
        if token[i:i + 1] == "<" and i + 5 <= len(token) and token[i:i + 3] == "<0x" and token[i + 5] == ">":
            hex_str = token[i + 3:i + 5]
            result.append(int(hex_str, 16))
            i += 6
        else:
            result.append(ord(token[i]))
            i += 1
    return bytes(result)


@staticmethod
def _corpus_to_words(text: str) -> list[list[str]]:
    """Split corpus into whitespace-delimited words, each as a list of byte tokens."""
    words: list[list[str]] = []
    for word in text.split():
        byte_tokens = [BPETokenizer._byte_token(b) for b in word.encode("utf-8")]
        if byte_tokens:
            words.append(byte_tokens)
    return words


@staticmethod
def _count_pairs(words: list[list[str]]) -> Counter:
    """Count adjacent token pairs across all words."""
    counts: Counter = Counter()
    for word in words:
        for i in range(len(word) - 1):
            counts[(word[i], word[i + 1])] += 1
    return counts


@staticmethod
def _apply_merge(
    words: list[list[str]],
    pair: tuple[str, str],
    merged: str,
) -> list[list[str]]:
    """Replace all occurrences of *pair* in *words* with *merged*."""
    new_words: list[list[str]] = []
    for word in words:
        new_words.append(BPETokenizer._apply_merge_to_sequence(word, pair[0], pair[1], merged))
    return new_words


@staticmethod
def _apply_merge_to_sequence(
    tokens: list[str],
    left: str,
    right: str,
    merged: str | None = None,
) -> list[str]:
    """Merge adjacent *left*+*right* tokens into *merged* in a token list."""
    if merged is None:
        merged = left + right
    result: list[str] = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == left and tokens[i + 1] == right:
            result.append(merged)
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result


# Attach helpers to the class
BPETokenizer._byte_token = _byte_token
BPETokenizer._token_to_bytes = _token_to_bytes
BPETokenizer._corpus_to_words = _corpus_to_words
BPETokenizer._count_pairs = _count_pairs
BPETokenizer._apply_merge = _apply_merge
BPETokenizer._apply_merge_to_sequence = _apply_merge_to_sequence

### `train` — Learn BPE Merge Rules

The training algorithm:

1. Reset vocabulary and seed special tokens (IDs 0–3) + 256 base byte tokens (IDs 4–259).
2. Convert the corpus into whitespace-delimited words, each represented as a list of byte tokens.
3. Repeat until the vocabulary reaches the target size:
   - Count all adjacent token pairs across all words.
   - Find the most frequent pair.
   - Merge that pair into a new token and add it to the vocabulary.
   - Replace all occurrences of the pair in the word lists.

In [ ]:
def train(self, text: str) -> None:
    """Learn BPE merge rules from *text*.

    Populates ``self.merges`` and ``self.vocab`` so that the total
    vocabulary size equals ``self.vocab_size``.
    """
    # Reset state for a fresh training run
    self.vocab = Vocabulary()
    self.merges = []
    for token in self.SPECIAL_TOKENS:
        self.vocab.add_token(token)

    # Add the 256 base byte tokens
    for byte_val in range(256):
        self.vocab.add_token(self._byte_token(byte_val))

    # Convert corpus to list of "words" (each word is a list of byte tokens)
    words = self._corpus_to_words(text)

    # Iteratively merge the most frequent pair
    num_merges = self.vocab_size - len(self.vocab)
    for _ in range(num_merges):
        pair_counts = self._count_pairs(words)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        merged_token = best_pair[0] + best_pair[1]
        self.vocab.add_token(merged_token)
        self.merges.append(best_pair)
        words = self._apply_merge(words, best_pair, merged_token)


BPETokenizer.train = train

### `encode` — Text to Token IDs

Encoding converts a string to a list of integer token IDs:

1. Convert the input string to UTF-8 bytes.
2. Map each byte to its `<0xHH>` token.
3. Apply the learned merge rules in order — each merge combines adjacent
   tokens that match a learned pair.
4. Look up the final token strings in the vocabulary to get integer IDs.

An empty string returns an empty list.

In [ ]:
def encode(self, text: str) -> list[int]:
    """Encode *text* into a list of token IDs using learned merges."""
    if not text:
        return []

    tokens = [self._byte_token(b) for b in text.encode("utf-8")]

    # Apply merges in learned order
    for left, right in self.merges:
        tokens = self._apply_merge_to_sequence(tokens, left, right)

    return [self.vocab.get_id(t) for t in tokens]


BPETokenizer.encode = encode

### `decode` — Token IDs to Text

Decoding reverses the encoding process:

1. Map each token ID back to its string token via the vocabulary.
2. Convert each token string back to raw bytes by parsing `<0xHH>` sequences.
3. Concatenate all bytes and decode as UTF-8.

Special tokens (`<pad>`, `<bos>`, `<eos>`, `<unk>`) are emitted as the
literal string `<unk>` in the output. Unknown IDs also map to `<unk>`.

In [ ]:
def decode(self, token_ids: list[int]) -> str:
    """Decode a list of token IDs back to a text string.

    Unknown IDs are replaced with ``<unk>``.
    """
    tokens = [self.vocab.get_token(tid) for tid in token_ids]
    raw_bytes = bytearray()
    for token in tokens:
        if token in self.SPECIAL_TOKENS:
            # Special tokens are not byte-representable; emit <unk> bytes
            raw_bytes.extend("<unk>".encode("utf-8"))
            continue
        raw_bytes.extend(self._token_to_bytes(token))
    return raw_bytes.decode("utf-8", errors="replace")


BPETokenizer.decode = decode

### `save` / `load` — Persistence

The tokenizer state (vocabulary mapping and merge list) is serialized to a
JSON file. The format:

```json
{
  "token_to_id": {"<pad>": 0, "<bos>": 1, ...},
  "merges": [["<0x48>", "<0x65>"], ...]
}
```

Loading reconstructs the `Vocabulary` and merge list from this file.

In [ ]:
def save(self, path: str) -> None:
    """Save vocabulary and merges to a JSON file at *path*."""
    data: dict[str, Any] = {
        "token_to_id": self.vocab.token_to_id,
        "merges": [list(pair) for pair in self.merges],
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load(self, path: str) -> None:
    """Load vocabulary and merges from a JSON file at *path*.

    Raises:
        FileNotFoundError: If *path* does not exist.
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    self.vocab = Vocabulary()
    for token, tid in sorted(data["token_to_id"].items(), key=lambda x: x[1]):
        added_id = self.vocab.add_token(token)
        assert added_id == tid, f"ID mismatch for {token!r}: {added_id} != {tid}"

    self.merges = [tuple(pair) for pair in data["merges"]]


BPETokenizer.save = save
BPETokenizer.load = load

---

## Example Usage

The cells below demonstrate training a small BPE tokenizer on sample text
(English and Arabic), then encoding and decoding strings.

### Demo: Training on Sample Text

We train a small tokenizer (vocab size 300) on a short English + Arabic corpus.
After training, we inspect the vocabulary size and the first few learned merges.

In [ ]:
sample_corpus = (
    "the cat sat on the mat the cat sat on the mat "
    "the dog sat on the log the dog sat on the log "
    "hello world hello world hello world "
    "\u0645\u0631\u062d\u0628\u0627 \u0628\u0627\u0644\u0639\u0627\u0644\u0645 "  # مرحبا بالعالم
    "\u0627\u0644\u0633\u0644\u0627\u0645 \u0639\u0644\u064a\u0643\u0645 "  # السلام عليكم
    "\u0643\u064a\u0641 \u062d\u0627\u0644\u0643 "  # كيف حالك
)

tokenizer = BPETokenizer(vocab_size=300)
tokenizer.train(sample_corpus)

print(f"Vocabulary size: {len(tokenizer.vocab)}")
print(f"Number of merges learned: {len(tokenizer.merges)}")
print(f"\nFirst 10 merges:")
for i, (left, right) in enumerate(tokenizer.merges[:10]):
    print(f"  {i}: {left!r} + {right!r} -> {left + right!r}")

### Demo: Encode and Decode (English)

Encode a simple English string, inspect the token IDs, then decode back.
The round-trip should produce the original string.

In [ ]:
text_en = "the cat sat"
ids = tokenizer.encode(text_en)
decoded = tokenizer.decode(ids)

print(f"Original:  {text_en!r}")
print(f"Token IDs: {ids}")
print(f"Decoded:   {decoded!r}")
print(f"Round-trip match: {text_en == decoded}")

### Demo: Encode and Decode (Arabic)

Arabic characters are multi-byte in UTF-8 (typically 2 bytes each).
The byte-level BPE handles them correctly — merges learn common Arabic
byte sequences without splitting within a code point.

In [ ]:
text_ar = "\u0645\u0631\u062d\u0628\u0627 \u0628\u0627\u0644\u0639\u0627\u0644\u0645"  # مرحبا بالعالم
ids_ar = tokenizer.encode(text_ar)
decoded_ar = tokenizer.decode(ids_ar)

print(f"Original:  {text_ar}")
print(f"Token IDs: {ids_ar}")
print(f"Decoded:   {decoded_ar}")
print(f"Round-trip match: {text_ar == decoded_ar}")

### Demo: Mixed Language Text

The tokenizer handles mixed English and Arabic text seamlessly.

In [ ]:
text_mixed = "Hello \u0645\u0631\u062d\u0628\u0627 World \u0627\u0644\u0639\u0627\u0644\u0645"
ids_mixed = tokenizer.encode(text_mixed)
decoded_mixed = tokenizer.decode(ids_mixed)

print(f"Original:  {text_mixed}")
print(f"Token IDs: {ids_mixed}")
print(f"Decoded:   {decoded_mixed}")
print(f"Round-trip match: {text_mixed == decoded_mixed}")

### Demo: Save and Load

Save the trained tokenizer to a temporary JSON file, load it into a fresh
instance, and verify that encoding produces the same result.

In [ ]:
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    tmp_path = f.name

tokenizer.save(tmp_path)
print(f"Saved tokenizer to {tmp_path}")

# Load into a fresh instance
tokenizer2 = BPETokenizer(vocab_size=300)
tokenizer2.load(tmp_path)
os.unlink(tmp_path)

# Verify same encoding
test_text = "the cat sat"
ids_original = tokenizer.encode(test_text)
ids_loaded = tokenizer2.encode(test_text)

print(f"Original tokenizer IDs: {ids_original}")
print(f"Loaded tokenizer IDs:   {ids_loaded}")
print(f"Match: {ids_original == ids_loaded}")

### Demo: Vocabulary Inspection

Inspect the `Vocabulary` object directly — check bijectivity and special tokens.

In [ ]:
vocab = tokenizer.vocab
print(f"Vocabulary size: {len(vocab)}")
print(f"token_to_id size: {len(vocab.token_to_id)}")
print(f"id_to_token size: {len(vocab.id_to_token)}")
print(f"Bijective: {len(vocab.token_to_id) == len(vocab.id_to_token)}")

# Special tokens present
for name, expected_id in BPETokenizer.SPECIAL_TOKENS.items():
    actual_id = vocab.get_id(name)
    print(f"  {name} -> ID {actual_id} (expected {expected_id})")

# Unknown token fallback
print(f"\nUnknown token lookup: vocab.get_id('NONEXISTENT') = {vocab.get_id('NONEXISTENT')}")
print(f"Unknown ID lookup: vocab.get_token(999999) = {vocab.get_token(999999)!r}")